# 03 — Modo Comparativo

**Objetivo**: comparar os quatro algoritmos de clustering (K-Means, DBSCAN,
Aglomerativo e Mean Shift) no mesmo dataset, coletando métricas de avaliação internas
e externas, e gerando visualizações lado a lado e uma tabela de ranking.

## Tópicos
1. Pipeline unificado de pré-processamento
2. Execução de todos os algoritmos
3. Métricas internas (silhouette, Davies-Bouldin, Calinski-Harabász)
4. Métricas externas (ARI, NMI, V-measure) — quando ground-truth disponível
5. Tabela de ranking
6. Gráfico de barras das métricas
7. Scatter PCA 2D lado a lado

In [ ]:
# Imports do projeto
from ml_clustering_lab.datasets.loaders import load_sklearn
from ml_clustering_lab.preprocessing.cleaning import drop_missing, drop_duplicates
from ml_clustering_lab.preprocessing.feature_selection import select_numeric_features
from ml_clustering_lab.preprocessing.scaling import scale_features
from ml_clustering_lab.clustering import ALGORITHM_REGISTRY, get_algorithm
from ml_clustering_lab.clustering.evaluation import (
    compute_internal_metrics,
    compute_external_metrics,
    build_comparison_table,
)
from ml_clustering_lab.visualization.embeddings import reduce_pca

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
sns.set_theme(style='whitegrid')

print('Algoritmos disponíveis:', list(ALGORITHM_REGISTRY.keys()))

## 1. Pré-processamento unificado

In [ ]:
df = load_sklearn('iris')

# Limpeza: remover nulos e duplicados primeiro, manter alinhamento com target
df_clean = drop_missing(drop_duplicates(df))
true_labels = df_clean['target'].values  # ground-truth alinhado ao df limpo

# Features e escalonamento
X_df = select_numeric_features(df_clean, exclude=['target'])
X_scaled = scale_features(X_df).values
X_2d = reduce_pca(X_scaled)

print(f'Shape: {X_scaled.shape}  |  PCA 2D: {X_2d.shape}')

## 2. Execução de todos os algoritmos

In [ ]:
# Hiperparâmetros por algoritmo
algo_configs = {
    'kmeans':        {'n_clusters': 3},
    'dbscan':        {'eps': 0.5, 'min_samples': 5},
    'agglomerative': {'n_clusters': 3, 'linkage': 'ward'},
    'mean-shift':    {},
}

results = {}
for name, kwargs in algo_configs.items():
    algo = get_algorithm(name, **kwargs)
    t0 = time.time()
    labels = algo.fit_predict(X_scaled)
    elapsed = time.time() - t0
    results[name] = {'labels': labels, 'elapsed': elapsed, 'algo': algo}
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    print(f'{algo.name:20s} → {n_clusters} clusters  ({elapsed*1000:.1f} ms)')

## 3 & 4. Coleta de métricas internas e externas

In [ ]:
rows = []
for name, res in results.items():
    labels = res['labels']
    internal = compute_internal_metrics(X_scaled, labels)
    external = compute_external_metrics(labels, true_labels)

    row = {
        'algorithm': res['algo'].name,
        'elapsed_time': res['elapsed'],
        **internal,
        **{f'ext_{k}': v for k, v in external.items()},
    }
    rows.append(row)

df_metrics = pd.DataFrame(rows).set_index('algorithm')
df_metrics.round(4)

## 5. Tabela de ranking (por silhouette)

In [ ]:
comparison_table = build_comparison_table(rows)
comparison_table.set_index('algorithm').round(4)

## 6. Gráfico de barras das métricas

In [ ]:
metrics_to_plot = {
    'silhouette':        'Silhouette ↑ (maior = melhor)',
    'davies_bouldin':    'Davies-Bouldin ↓ (menor = melhor)',
    'calinski_harabasz': 'Calinski-Harabász ↑ (maior = melhor)',
}

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
palette = sns.color_palette('tab10', len(results))

for ax, (metric, label) in zip(axes, metrics_to_plot.items()):
    vals = df_metrics[metric]
    vals.plot.bar(ax=ax, color=palette, edgecolor='white', width=0.7)
    ax.set_title(label, fontsize=11)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=30)

plt.suptitle('Comparação de métricas — Iris', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Scatter PCA 2D lado a lado

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, (name, res) in zip(axes, results.items()):
    labels = res['labels']
    sc = ax.scatter(X_2d[:, 0], X_2d[:, 1], c=labels, cmap='tab10', s=30, alpha=0.85)
    ax.set_title(res['algo'].name)
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')
    plt.colorbar(sc, ax=ax)

plt.suptitle('Comparação visual dos clusters — PCA 2D — Iris', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Novos cenários sintéticos

Usamos o `run_compare` do pipeline para comparar todos os algoritmos
nos 6 cenários sintéticos disponíveis.
Isso revela os pontos fortes e fracos de cada algoritmo de forma visual.


In [ ]:
from ml_clustering_lab.datasets import generate, AVAILABLE_SCENARIOS
from ml_clustering_lab.pipeline.run_compare import run_compare

scenario_results = {}
for scenario in AVAILABLE_SCENARIOS:
    print(f'Comparando algoritmos em: {scenario}...')
    df_cmp = run_compare(
        algorithms=['kmeans', 'dbscan', 'agglomerative'],
        dataset=scenario,
        n_clusters=3,
        outdir=f'outputs/runs/compare_{scenario}',
    )
    scenario_results[scenario] = df_cmp
    print(df_cmp[['algorithm', 'silhouette', 'n_clusters']].to_string(index=False))
    print()


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Construir tabela resumo: silhouette por (cenário, algoritmo)
records = []
for scenario, df_cmp in scenario_results.items():
    for _, row in df_cmp.iterrows():
        records.append({'scenario': scenario, 'algorithm': row['algorithm'], 'silhouette': row.get('silhouette', float('nan'))})

summary_df = pd.DataFrame(records).pivot(index='algorithm', columns='scenario', values='silhouette')
summary_df = summary_df[AVAILABLE_SCENARIOS]  # preserve order

fig, ax = plt.subplots(figsize=(12, 4))
summary_df.T.plot(kind='bar', ax=ax, width=0.7)
ax.set_xlabel('Cenário')
ax.set_ylabel('Silhouette Score')
ax.set_title('Silhouette Score por algoritmo e cenário')
ax.legend(title='Algoritmo', bbox_to_anchor=(1.01, 1), loc='upper left')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

print('\nTabela completa:')
print(summary_df.round(4))
